# Weak Gravitational Lensing Fundamentals

This notebook covers the theoretical background for galaxy shape measurement
in the context of weak gravitational lensing.

**Key reference:** Mandelbaum (2018), *Weak Lensing for Precision Cosmology*, [arXiv:1710.03235](https://arxiv.org/abs/1710.03235)

## 1. The Lensing Effect

Light from distant (background) galaxies is deflected by the gravitational
field of intervening (foreground) matter. In the **weak lensing** regime,
this produces a small but coherent distortion of galaxy shapes.

The lensing is described by the **convergence** $\kappa$ (magnification)
and **shear** $\gamma = \gamma_1 + i\gamma_2$ (shape distortion).

For a circular source, lensing transforms it into an ellipse with
ellipticity:

$$e = \frac{a - b}{a + b}$$

where $a$ and $b$ are the semi-major and semi-minor axes.

In practice, galaxies have **intrinsic shapes**, so the observed ellipticity is:

$$e^{\text{obs}} = e^{\text{intrinsic}} + \gamma$$

(in the weak lensing limit, $|\gamma| \ll 1$)

## 2. Why Shapes Are Hard to Measure

The typical weak lensing shear is $|\gamma| \sim 0.01$–$0.03$, while
the intrinsic ellipticity dispersion is $\sigma_e \sim 0.25$–$0.3$.
We need many galaxies to beat down the **shape noise**:

$$\sigma_\gamma = \frac{\sigma_e}{\sqrt{N_{\text{gal}}}}$$

Additional challenges:
- **PSF smearing:** The point spread function circularizes galaxies → must be corrected
- **PSF anisotropy:** An elliptical PSF mimics a shear signal → must be modeled
- **Noise bias:** Pixel noise biases shape estimators (multiplicative bias)
- **Model bias:** Assuming the wrong galaxy profile biases the result
- **Blending:** Overlapping galaxies contaminate shape measurements
- **Selection bias:** Preferentially selecting round galaxies biases the shear

## 3. Shape Measurement Methods

### 3.1 Moments-based: KSB
Kaiser, Squires & Broadhurst (1995) — measure weighted second moments
of the surface brightness and correct for PSF using a "smear polarizability" tensor.

### 3.2 Re-Gaussianization (REGAUSS)
Hirata & Seljak (2003) — the default in the LSST/HSC pipeline.
Measures adaptive moments (like SExtractor's `FLUX_RADIUS`), then
applies a correction for non-Gaussianity of the PSF.

### 3.3 Model-fitting
Fit a parametric galaxy model (e.g., Sérsic profile) convolved with the PSF
to the image. Examples: im3shape, lensfit.

### 3.4 Metacalibration
Sheldon & Huff (2017) — self-calibrating method that estimates the
**response** of a shape estimator to shear by artificially shearing the
image and re-measuring. Corrects multiplicative bias without simulations.

### 3.5 Available in LSST Pipelines (`lsst.meas.extensions.shapeHSM`)
| Plugin | Method |
|--------|--------|
| `HsmShapeKsb` | KSB |
| `HsmShapeRegauss` | REGAUSS |
| `HsmShapeBj` | Bernstein & Jarvis |
| `HsmShapeLinear` | Linear (modified BJ) |

## 4. Ellipticity Definitions

Two common conventions:

**Distortion** (used by KSB):
$$e = \frac{a^2 - b^2}{a^2 + b^2}, \quad e_1 = e\cos(2\phi), \quad e_2 = e\sin(2\phi)$$

**Shear-type** (used by REGAUSS, BJ):
$$\epsilon = \frac{a - b}{a + b}, \quad \epsilon_1 = \epsilon\cos(2\phi), \quad \epsilon_2 = \epsilon\sin(2\phi)$$

Both relate to the **quadrupole moments** $Q_{ij}$ of the brightness distribution:

$$e_1 = \frac{Q_{xx} - Q_{yy}}{Q_{xx} + Q_{yy}}, \quad e_2 = \frac{2Q_{xy}}{Q_{xx} + Q_{yy}}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse


def draw_ellipse(ax, e1, e2, x=0, y=0, size=1.0, **kwargs):
    """Draw an ellipse with given ellipticity components."""
    e = np.sqrt(e1**2 + e2**2)
    phi = 0.5 * np.arctan2(e2, e1)
    q = (1 - e) / (1 + e)  # axis ratio
    a = size
    b = size * q
    ellipse = Ellipse(
        (x, y), width=2*a, height=2*b,
        angle=np.degrees(phi), fill=False, **kwargs
    )
    ax.add_patch(ellipse)


fig, axes = plt.subplots(1, 3, figsize=(12, 4))

# Circular (no shear)
ax = axes[0]
draw_ellipse(ax, 0, 0, size=0.8, lw=2, color='steelblue')
ax.set_title('No shear: $e_1=0, e_2=0$')
ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5)
ax.set_aspect('equal')

# Positive e1 shear
ax = axes[1]
draw_ellipse(ax, 0.3, 0, size=0.8, lw=2, color='crimson')
ax.set_title('$\gamma_1 = 0.3$: stretched along x')
ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5)
ax.set_aspect('equal')

# Positive e2 shear
ax = axes[2]
draw_ellipse(ax, 0, 0.3, size=0.8, lw=2, color='forestgreen')
ax.set_title('$\gamma_2 = 0.3$: stretched at 45°')
ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5)
ax.set_aspect('equal')

plt.tight_layout()
plt.savefig('../../figures/ellipticity_demo.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. The Pipeline Chain

```
Raw images
  │
  ▼
ISR (bias, dark, flat, crosstalk, brighter-fatter)
  │
  ▼
PSF modeling (PIFF / PSFEx) ← critical for shape measurement
  │
  ▼
Astrometric + photometric calibration
  │
  ▼
Coaddition (warp + stack exposures)
  │
  ▼
Detection + deblending (scarlet)
  │
  ▼
Shape measurement (HSM: REGAUSS, KSB, ...)
  │
  ▼
Photometry (CModel) → colours → photo-z (RAIL)
  │
  ▼
Shear catalog → weak lensing science
```

## Next Steps

- **Notebook 02:** Explore the LSST pipeline Butler and data access
- **Notebook 03:** Hands-on shape measurement with GalSim simulations
- **Notebook 04:** PSF modeling and correction